In [ ]:
#| hide
from kavacha import *


# kavacha

kavacha wraps a local web app in a native window and packages it for macOS or Windows. It uses py2app and py2exe as its freezers.

## Install

```sh
pip install kavacha
pip install "kavacha[window]"
pip install "kavacha[macapp]"
pip install "kavacha[windows]"
```

The extras add native-window support and platform freezer dependencies.

## Describe the app

```python
from kavacha import App, tree, doc_types, mypyc_modules

app = App(
    name='Demo',
    entry='demo_app.py',
    version='1.2.3',
    icon='assets/Demo.icns',
    packages=['demo', 'fasthtml', 'uvicorn', 'numpy'],
    includes=['demo.cli', *mypyc_modules()],
    grafted=['apsw', 'playwright'],
    data=tree('demo/static', 'demo/static'),
    extras=['desktop'],
    doc_types=doc_types(['.py', '.rs', '.md']),
)
app.py2app_options()
```

One `App` specification supplies both platform freezers. Explicit packages and modules cover imports a freezer cannot discover.

## Build

```python
from kavacha import build, check

check(app, root)
build(app, root, setup_py='packaging/macos/setup.py')
```

`build` uses a compatible interpreter, installs pinned requirements from `uv.lock`, refuses to replace a running bundle unless forced, and records the source commit.

## After freezing

```python
from kavacha import finish

finish(bundle, app)
```

`finish` grafts archive-incompatible packages, removes unreachable archive copies, links duplicate files, and installs the modern icon before signing.

## Icons

```python
from kavacha.icons import icons_for

icons_for('assets/logo.png', 'assets', 'Demo')
```

A square image of at least 512 pixels produces platform and web icon assets. Invalid sources are rejected rather than upscaled.

## Window

```python
from kavacha.window import run_shell, shell_ready

ok, why = shell_ready()
run_shell('http://127.0.0.1:8000', title='Demo', size='1440x900')
```

`run_shell` opens a pywebview window over a loopback server. The host supplies menus and workspace callbacks.

## Status

The spec, build driver, bundle surgery and icons are covered by tests that run anywhere. The window
and menus are ported working code, but the platform they matter on is not one CI can exercise here
— treat that module as needing a real run on macOS before you rely on it.